### Setup

Environment setup and exploratory analysis

In [ ]:
import sys
import platform
import numpy as np
import pandas as pd
import scipy
import scanpy as sc

print("Python:", sys.version)
print("Executable:", sys.executable)
print("Platform:", platform.platform())
print("NumPy:", np.__version__)
print("Pandas:", pd.__version__)
print("SciPy:", scipy.__version__)
print("Scanpy:", sc.__version__)

/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_csv from `anndata` is deprecated. Import anndata.io.read_csv instead.
  warnings.warn(msg, FutureWarning)
/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_excel from `anndata` is deprecated. Import anndata.io.read_excel instead.
  warnings.warn(msg, FutureWarning)
/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_hdf from `anndata` is deprecated. Import anndata.io.read_hdf instead.
  warnings.warn(msg, FutureWarning)
/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_loom from `anndata` is deprecated. Import anndata.io.read_loom instead.
  warnings.warn(msg, FutureWarning)
/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:434: FutureWarning: Importing read_mtx from `anndata` i

Python: 3.10.13 (main, Sep 19 2023, 15:54:50) [GCC 12.3.1 20230526]
Executable: /home/akash1/scgpt-cna/.venv/bin/python
Platform: Linux-5.14.0-570.44.1.el9_6.x86_64-x86_64-Intel-R-_Xeon-R-_6972P-with-glibc2.37
NumPy: 2.1.1
Pandas: 2.2.3
SciPy: 1.15.1
Scanpy: 1.10.3


In [ ]:
from pathlib import Path

DATA_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix"
)

assert DATA_DIR.exists()
print(DATA_DIR)

/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/benchmarking_spatial_scale/GDC-data/C3N-00148/C3N-00148-01/single_cell/dbe28c52-cdbf-477e-a728-9a61fd4a8139/qc_filtered_bc_feature_matrix


In [ ]:
from pathlib import Path

import anndata as ad
import pandas as pd
from scipy import sparse
from scipy.io import mmread

DATA_DIR = Path(
    "/home/akash1/projects/rrg-tperkins/dgladish/Projects/MSc/"
    "benchmarking_spatial_scale/GDC-data/"
    "C3N-00148/C3N-00148-01/single_cell/"
    "dbe28c52-cdbf-477e-a728-9a61fd4a8139/"
    "qc_filtered_bc_feature_matrix"
)

features = pd.read_csv(
    DATA_DIR / "features.tsv",
    sep="\t",
    header=None,
    names=["gene_id", "gene_symbol", "feature_type"],
    dtype=str,
)

barcodes = pd.read_csv(
    DATA_DIR / "barcodes.tsv",
    sep="\t",
    header=None,
    names=["barcode"],
    dtype=str,
)

# Matrix Market file is genes × cells.
# AnnData convention is cells × genes, so transpose it.
counts = mmread(DATA_DIR / "matrix.mtx").T.tocsr()

assert sparse.issparse(counts)
assert counts.shape == (len(barcodes), len(features))

adata = ad.AnnData(X=counts)

adata.obs_names = barcodes["barcode"].to_numpy()
adata.var_names = features["gene_symbol"].to_numpy()

adata.var["gene_id"] = features["gene_id"].to_numpy()
adata.var["feature_type"] = features["feature_type"].to_numpy()

# Some gene symbols may be duplicated.
adata.var_names_make_unique()

/home/akash1/scgpt-cna/.venv/lib/python3.10/site-packages/anndata/utils.py:268: UserWarning: Suffix used (-[0-9]+) to deduplicate index values may make index values difficult to interpret. There values with a similar suffixes in the index. Consider using a different delimiter by passing `join={delimiter}`Example key collisions generated by the make_index_unique algorithm: ['SNORD116-1', 'SNORD116-2', 'SNORD116-3', 'SNORD116-4', 'SNORD116-5']
  warnings.warn(


In [ ]:
print(adata)
print("Shape:", adata.shape)
print("Sparse:", sparse.issparse(adata.X))
print("Nonzero entries:", adata.X.nnz)
print(adata.var.head())
print(adata.obs.head())

AnnData object with n_obs × n_vars = 7598 × 60660
    var: 'gene_id', 'feature_type'
Shape: (7598, 60660)
Sparse: True
Nonzero entries: 12599909
                       gene_id     feature_type
DDX11L1      ENSG00000223972.5  Gene Expression
WASH7P       ENSG00000227232.5  Gene Expression
MIR6859-1    ENSG00000278267.1  Gene Expression
MIR1302-2HG  ENSG00000243485.5  Gene Expression
MIR1302-2    ENSG00000284332.1  Gene Expression
Empty DataFrame
Columns: []
Index: [AAACCCAAGAAGTGTT-1, AAACCCAAGCCTGTCG-1, AAACCCAAGCGCTGCT-1, AAACCCAAGTTCCGGC-1, AAACCCACAAGCGCTC-1]


In [ ]:
df = pd.read_csv(DATA_DIR / "features.tsv", sep="\t")
features.head()

,gene_id,gene_symbol,feature_type
0,ENSG00000223972.5,DDX11L1,Gene Expression
1,ENSG00000227232.5,WASH7P,Gene Expression
2,ENSG00000278267.1,MIR6859-1,Gene Expression
3,ENSG00000243485.5,MIR1302-2HG,Gene Expression
4,ENSG00000284332.1,MIR1302-2,Gene Expression
